# Basic Pinch Analysis
This notebook shows how the pinch, utility targets, and graph shapes move when the minimum approach temperature assumption changes.


## Case context
The packaged `crude_preheat_train.json` case represents a small crude-unit preheat train with named hot and cold duties. The workflow is: solve the baseline case, inspect the main graphs, then rerun the case after multiplying every stream and utility `dt_cont` value by the same factor.


In [21]:
from copy import deepcopy
import json
import numpy as np
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
from plotly.subplots import make_subplots

from OpenPinch import PinchProblem
from OpenPinch.resources import copy_sample_case

PLOT_WIDTH = 720
PLOT_HEIGHT = 540


def display_plotly(fig, *, width: int = PLOT_WIDTH, height: int = PLOT_HEIGHT):
    fig = fig.to_dict()
    figure = go.Figure(fig)
    figure.update_layout(width=width, height=height, autosize=False)
    display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))


workspace = TemporaryDirectory()
work_dir = Path(workspace.name)
case_path = copy_sample_case(
    "crude_preheat_train.json",
    work_dir / "crude_preheat_train.json",
)
problem = PinchProblem(problem_filepath=case_path)
summary = problem.summary_frame()
base_row = summary.loc[
    summary["Target"] == "crude_preheat_train/Direct Integration",
    [
        "Hot Utility Target",
        "Cold Utility Target",
        "Heat Recovery",
        "Hot Pinch",
        "Cold Pinch",
    ],
]
base_row


,Hot Utility Target,Cold Utility Target,Heat Recovery,Hot Pinch,Cold Pinch
0,749.999995,1000.0,5150.000005,NaN,145.0


## Baseline graphs
Render the baseline composite curve, shifted composite curve, and grand composite curve directly in the notebook so the graph shapes can be read before changing `dt_cont`.


In [22]:
composite_curve = problem.plot_composite_curve()
display_plotly(composite_curve)


In [23]:
shifted_composite_curve = problem.plot_composite_curve(variant="shifted")
display_plotly(shifted_composite_curve)


In [24]:
grand_composite_curve = problem.plot_grand_composite_curve()
display_plotly(grand_composite_curve)


In [25]:
base_payload = json.loads(case_path.read_text(encoding="utf-8"))
multipliers = np.linspace(0.5, 3.0, 26)


def _scale_dt_cont(value, factor: float):
    if value is None:
        return None
    if isinstance(value, dict):
        updated = dict(value)
        if updated.get("value") is not None:
            updated["value"] = float(updated["value"]) * factor
        return updated
    return float(value) * factor


rows = []
for factor in multipliers:
    payload = deepcopy(base_payload)
    for stream in payload["streams"]:
        stream["dt_cont"] = _scale_dt_cont(stream.get("dt_cont"), factor)
    for utility in payload.get("utilities", []):
        utility["dt_cont"] = _scale_dt_cont(utility.get("dt_cont"), factor)

    variant_path = work_dir / f"crude_preheat_train_dt_{factor:.1f}.json"
    variant_path.write_text(json.dumps(payload), encoding="utf-8")

    variant_problem = PinchProblem(problem_filepath=variant_path)
    variant_summary = variant_problem.summary_frame()
    row = variant_summary.loc[
        variant_summary["Target"] == f"{variant_path.stem}/Direct Integration"
    ].iloc[0]
    rows.append(
        {
            "dt_cont multiplier": factor,
            "Hot Utility Target": row["Hot Utility Target"],
            "Cold Utility Target": row["Cold Utility Target"],
            "Heat Recovery": row["Heat Recovery"],
            "Hot Pinch": row["Hot Pinch"],
            "Cold Pinch": row["Cold Pinch"],
        }
    )

sensitivity = pd.DataFrame(rows)
sensitivity


,dt_cont multiplier,Hot Utility Target,Cold Utility Target,Heat Recovery,Hot Pinch,Cold Pinch
0,0.5,549.999995,800.0,5350.000005,NaN,142.5
1,0.6,589.999995,840.0,5310.000005,NaN,143.0
2,0.7,629.999995,880.0,5270.000005,NaN,143.5
3,0.8,669.999995,920.0,5230.000005,NaN,144.0
4,0.9,709.999995,960.0,5190.000005,NaN,144.5
5,1.0,749.999995,1000.0,5150.000005,NaN,145.0
6,1.1,789.999995,1040.0,5110.000005,NaN,145.5
7,1.2,829.999995,1080.0,5070.000005,NaN,146.0
8,1.3,869.999995,1120.0,5030.000005,NaN,146.5
9,1.4,909.999995,1160.0,4990.000005,NaN,147.0


In [26]:
sensitivity_fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        "Utility Targets and Heat Recovery",
        "Pinch Temperatures",
    ),
)

for column in ["Hot Utility Target", "Cold Utility Target", "Heat Recovery"]:
    sensitivity_fig.add_trace(
        go.Scatter(
            x=sensitivity["dt_cont multiplier"],
            y=sensitivity[column],
            mode="lines+markers",
            name=column,
        ),
        row=1,
        col=1,
    )

for column in ["Hot Pinch", "Cold Pinch"]:
    sensitivity_fig.add_trace(
        go.Scatter(
            x=sensitivity["dt_cont multiplier"],
            y=sensitivity[column],
            mode="lines+markers",
            name=column,
        ),
        row=2,
        col=1,
    )

sensitivity_fig.update_xaxes(title_text="dt_cont multiplier", row=2, col=1)
sensitivity_fig.update_yaxes(title_text="kW or degC", row=1, col=1)
sensitivity_fig.update_yaxes(title_text="degC", row=2, col=1)
sensitivity_fig.update_layout(title="dt_cont sensitivity overview")
display_plotly(sensitivity_fig, height=700)
